# 05. Xem kết quả sau khi chạy các thí nghiệm E

Notebook gồm hai phần, chạy từ trên xuống:

| Phần | Nội dung | Cần GPU? |
|---|---|---|
| **A** | Nạp kết quả đã chạy, bảng tổng hợp, so sánh theo cặp, đường cong huấn luyện, biểu đồ so sánh | không (chạy được cả trên máy cá nhân) |
| **B** | Số tham số, GFLOPs, độ trễ; đánh giá trên tập test; ảnh dự đoán và ca khó | có (Kaggle GPU T4) |

**Đầu vào:** các file `E0.zip`, `E_main.zip`, `E1.zip`, `E2.zip` (và `E3.zip` nếu có) do `ssr_lib.nen_ket_qua` tạo sau mỗi lần chạy, hoặc thư mục đã giải nén.

**Cách đưa kết quả vào:**
- *Trên Kaggle:* tải 4 file zip lên một Dataset mới (Create → New Dataset), rồi Add Input vào notebook này. Hoặc Add Input trực tiếp output của notebook 03 đã Save & Run All.
- *Trên máy cá nhân (VS Code):* đặt 4 file zip vào một thư mục, ví dụ `E:\btl_thigiacmay\ket_qua_tai_ve`, rồi sửa `GOC_KET_QUA` ở cell dưới.

**Thiết lập phiên Kaggle:** phần A chỉ cần Accelerator = None. Phần B cần Accelerator = GPU T4, Internet = On, Add Input = `seedlingstemrow`.

## 0. Tải mã nguồn

In [ ]:
# Dùng cho cả Kaggle lẫn máy cá nhân
import os, sys, subprocess
GITHUB_USER = "Klark352"
TEN_REPO = "ssr-btl-thigiacmay"
if os.path.isdir("/kaggle"):
    DUONG_DAN = f"/kaggle/working/{TEN_REPO}"
    if not os.path.isdir(DUONG_DAN):
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{GITHUB_USER}/{TEN_REPO}.git", DUONG_DAN], check=True)
    else:
        subprocess.run(["git", "-C", DUONG_DAN, "pull"], check=True)
    SRC = f"{DUONG_DAN}/src"
else:                                   # máy cá nhân: notebook nằm trong thư mục notebooks của repo
    SRC = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, SRC)
import importlib, ket_qua as K
importlib.reload(K)
print("Mã nguồn sẵn sàng:", SRC)

## A. Xem kết quả (không cần GPU)

### A1. Nạp kết quả

Đặt `GOC_KET_QUA` là thư mục chứa các file zip (hoặc thư mục `runs`, `logs` đã giải nén). Để `None` thì tự tìm trong `/kaggle/input`.

In [ ]:
GOC_KET_QUA = None                      # ví dụ: r"E:\btl_thigiacmay\ket_qua_tai_ve" hoặc "/kaggle/input/ket-qua-thi-nghiem"
GHI_RA = "/kaggle/working/ket_qua" if os.path.isdir("/kaggle") else os.path.abspath("ket_qua")
GIAI_NEN = os.path.join(GHI_RA, "_giai_nen")      # nơi ghi được (Input trên Kaggle chỉ đọc)

if GOC_KET_QUA is None:
    ung_vien = ["/kaggle/working", "/kaggle/input"] if os.path.isdir("/kaggle") else [os.getcwd()]
else:
    ung_vien = [GOC_KET_QUA]
ds = {}
for g in ung_vien:
    ds = K.giai_nen_va_tim(g, GIAI_NEN)
    if ds:
        break
if not ds:
    raise SystemExit("Không tìm thấy kết quả. Kiểm tra lại GOC_KET_QUA hoặc Add Input.")

### A2. Bảng tổng hợp và so sánh theo cặp

Số liệu lấy từ lần kiểm định cuối cùng trong log (với `best.pt`, tập **val**). Số trên tập test ở phần B.

In [ ]:
bang = K.bang_tong_hop(ds)
cap = K.so_sanh_cap(bang)
display(bang)
display(cap)
K.luu(bang, cap, GHI_RA)
print(K.dinh_dang(bang))                # bảng markdown, dấu phẩy thập phân, dán thẳng vào báo cáo

**Cách đọc `so_sanh_cap`:** cột `Δ` là hiệu giữa mô hình thứ nhất và mô hình đối chứng, dương nghĩa là mô hình thứ nhất cao hơn.
Ba cặp tương ứng ba câu hỏi ở mục 3.4 của báo cáo: toàn bộ thành phần mới so với C2f (E_main so với E0), riêng DGST (E_main so với E2), và tốc độ học (E_main so với E1).
Hiệu nhỏ (vài phần nghìn) trên một lần chạy với một seed có thể chỉ là nhiễu, nên khi viết nhận xét cần nói rõ điều này.

### A3. Đường cong huấn luyện

In [ ]:
tep_hinh = K.ve_duong_cong(ds, GHI_RA)
K.bieu_do_so_sanh(bang, GHI_RA)

Cách đọc: loss giảm dần và các chỉ số tăng rồi đi ngang là dấu hiệu hội tụ. Nếu mAP50 vẫn còn tăng ở epoch cuối thì 20 epoch chưa đủ và thí nghiệm E3 (tinh chỉnh) có ý nghĩa.
Nếu chỉ số val dao động mạnh thì cần ghi nhận độ không ổn định, vì tập val chỉ có 179 ảnh.

### A4. Đoạn tóm tắt để gửi cho Claude điền báo cáo

In [ ]:
print("===== TÓM TẮT =====")
print(K.dinh_dang(bang))
print()
print(K.dinh_dang(cap))
for ten, f in ds.items():
    d, s = K.doc_log(f["log"])
    print(f"{ten}: {len(d)} lần kiểm định trong log; epoch mAP50 cao nhất theo csv = ",
          bang.loc[bang['Mã'] == ten, 'Epoch mAP50 cao nhất'].iloc[0] if 'Epoch mAP50 cao nhất' in bang.columns else None)

## B. Đánh giá cần GPU (Kaggle)

Phần này cần repo gốc YOLO-SSR, dữ liệu và `best.pt`. Chạy trong phiên có GPU. Mã gọi vào API của nhánh Ultralytics đa nhiệm của tác giả
và **chưa được kiểm thử trên repo thật**; mỗi bước in rõ lỗi nếu có. Nếu một bước lỗi, gửi nguyên văn thông báo lỗi để chỉnh.

### B0. Chuẩn bị môi trường và dữ liệu

In [ ]:
import setup_env, ssr_lib as L
importlib.reload(L)
setup_env.cai_dat()
try:
    L.chuan_bi_du_lieu()
except RuntimeError as loi:
    print("CẢNH BÁO:", loi)
L.tao_yaml_du_lieu()
import danh_gia_nang as N
importlib.reload(N)

### B1. Số tham số, GFLOPs và độ trễ

Đo với ảnh 640×640, batch = 1, chỉ tính lan truyền thuận. Độ trễ phụ thuộc phần cứng nên phải ghi rõ máy đo (GPU T4 và số lõi CPU của Kaggle) trong báo cáo.

In [ ]:
phuc_tap = N.do_phuc_tap(ds, imgsz=640, lan=100, luu_vao=GHI_RA)
display(phuc_tap)

### B2. Đánh giá trên tập test

Trước hết kiểm tra chéo: chạy lại trên tập **val** và so với số cuối trong log huấn luyện. Chỉ khi hai số khớp (độ lệch dưới 0,01) mới tin kết quả trên **test**.
Nếu `split="test"` bị lỗi, nhiều khả năng bộ kiểm định của repo cố định dùng tập val và cần sửa chỗ đọc `split`.

In [ ]:
N.kiem_tra_chay_val(ds, "E_main")

In [ ]:
bang_test = N.danh_gia_tat_ca(ds, split="test", luu_vao=GHI_RA)
display(bang_test)
print(K.dinh_dang(bang_test))

### B3. Ảnh dự đoán và ca khó

Chọn các ảnh test có nhiều thân cây nhất (dày đặc nên khó) cùng một số ảnh ngẫu nhiên, dự đoán bằng E_main và E0 để so sánh trực quan, từ đó tìm trường hợp thất bại cho mục 7.2 của báo cáo.

In [ ]:
anh = N.chon_anh_kho(so_anh=12)
for ten in ("E_main", "E0"):
    if ten in ds and ds[ten].get("best"):
        thu_muc = N.du_doan(ten, ds[ten]["best"], anh)
        if thu_muc:
            N.ghep_luoi(thu_muc, so_cot=3, luu_vao=f"{GHI_RA}/du_doan_{ten}.png")

### B4. Gói kết quả để tải về

In [ ]:
import zipfile
dich = f"{GHI_RA}.zip"
with zipfile.ZipFile(dich, "w", zipfile.ZIP_DEFLATED) as z:
    for goc, _, files in os.walk(GHI_RA):
        if "_giai_nen" in goc:
            continue
        for f in files:
            z.write(os.path.join(goc, f), os.path.relpath(os.path.join(goc, f), os.path.dirname(GHI_RA)))
print("Đã nén:", dich, round(os.path.getsize(dich) / 1e6, 1), "MB")